In [ ]:
!pip install -q transformers datasets scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from google.colab import drive

MODEL_NAME = "indobenchmark/indobert-base-p2"
LEARNING_RATE = 2e-5
BATCH_SIZE = 16
EPOCHS = 5
MAX_LENGTH = 128
RANDOM_STATE = 42

drive.mount('/content/drive')
path = '/content/drive/MyDrive/Skripsi 3,5 Tahun/skripsi_dataset/Revisi/DATASET_SIAP_LABEL_FINAL_CEK.csv'

df = pd.read_csv(path)
df['label'] = pd.to_numeric(df['label'], errors='coerce')
df = df.dropna(subset=['label', 'text_slang_removal']).reset_index(drop=True)
df['label'] = df['label'].astype(int)

print(f"Total data bersih: {len(df)}")
print(f"Distribusi label: {df['label'].value_counts().sort_index().to_dict()}")

df_train, df_temp = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE, stratify=df['label'])
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=df_temp['label'])

print(f"\nData Split:")
print(f"  Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
print(f"  Train dist: {df_train['label'].value_counts().sort_index().to_dict()}")
print(f"  Val dist:   {df_val['label'].value_counts().sort_index().to_dict()}")
print(f"  Test dist:  {df_test['label'].value_counts().sort_index().to_dict()}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text_slang_removal"], padding="max_length",
                     truncation=True, max_length=MAX_LENGTH)

train_ds = Dataset.from_pandas(df_train[['text_slang_removal', 'label']])
val_ds = Dataset.from_pandas(df_val[['text_slang_removal', 'label']])
test_ds = Dataset.from_pandas(df_test[['text_slang_removal', 'label']])

train_tokenized = train_ds.map(tokenize_function, batched=True)
val_tokenized = val_ds.map(tokenize_function, batched=True)
test_tokenized = test_ds.map(tokenize_function, batched=True)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir='./logs',
    logging_steps=10,
    save_total_limit=2,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
)

print("\nMemulai Fine-tuning IndoBERT...")
trainer.train()

print("\nMengevaluasi Data Uji...")
preds = trainer.predict(test_tokenized)
pred_labels = np.argmax(preds.predictions, axis=-1)
true_labels = test_tokenized['label']

print("\n" + "="*50)
print("           HASIL EVALUASI MODEL")
print("="*50)
print(classification_report(true_labels, pred_labels,
                          target_names=['Negatif', 'Netral', 'Positif'],
                          digits=4))

# ==================== CONFUSION MATRIX ====================
cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negatif', 'Netral', 'Positif'],
            yticklabels=['Negatif', 'Netral', 'Positif'],
            cbar_kws={'label': 'Jumlah'})
plt.xlabel('Prediksi Model', fontsize=12)
plt.ylabel('Label Sebenarnya', fontsize=12)
plt.title('Confusion Matrix - Analisis Sentimen Publik Terhadap Polri', fontsize=13, pad=15)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# ==================== LEARNING CURVE ====================
history = trainer.state.log_history

train_loss = [(x['epoch'], x['loss']) for x in history if 'loss' in x]
eval_metrics = [(x['epoch'], x['eval_loss']) for x in history if 'eval_loss' in x]

if train_loss and eval_metrics:
    epochs_train, losses_train = zip(*train_loss)
    epochs_eval, losses_eval = zip(*eval_metrics)

    plt.figure(figsize=(10, 6))
    plt.plot(epochs_train, losses_train, label='Training Loss', color='#3498db',
             linewidth=2, alpha=0.7, marker='o', markersize=4)
    plt.plot(epochs_eval, losses_eval, label='Validation Loss', color='#e74c3c',
             linewidth=2, marker='s', markersize=6)
    plt.title('Kurva Pembelajaran Model IndoBERT', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend(fontsize=11, loc='best')
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig('learning_curve.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(f"\nTraining Summary:")
    print(f"  Initial Loss: {losses_train[0]:.4f}")
    print(f"  Final Loss: {losses_train[-1]:.4f}")
    print(f"  Best Val Loss: {min(losses_eval):.4f} (Epoch {epochs_eval[losses_eval.index(min(losses_eval))]:.0f})")
else:
    print("Tidak dapat membuat learning curve - log history tidak lengkap")

# ==================== SAVE MODEL ====================
trainer.save_model("./model_sentimen_polri")
tokenizer.save_pretrained("./model_sentimen_polri")

print("\n" + "="*60)
print("TRAINING SELESAI!")
print("="*60)
print("\nFile yang dihasilkan:")
print("  1. confusion_matrix.png")
print("  2. learning_curve.png")
print("  3. ./model_sentimen_polri/ (model + tokenizer)")
print("\nGunakan untuk Bab IV skripsi!")
print("="*60)

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

print("Memuat 'Senjata' (Model 85% Akurasi)...")
model = AutoModelForSequenceClassification.from_pretrained("./model_sentimen_polri")
tokenizer = AutoTokenizer.from_pretrained("./model_sentimen_polri")
print("Model siap digunakan!\n")

path = '/content/drive/MyDrive/Skripsi 3,5 Tahun/skripsi_dataset/Revisi/DATASET_SIAP_LABEL_FINAL_CEK.csv'
df_asli = pd.read_csv(path)

df_labeled = df_asli.dropna(subset=['label']).copy()
df_unlabeled = df_asli[df_asli['label'].isna()].copy()

df_unlabeled = df_unlabeled.dropna(subset=['text_slang_removal']).reset_index(drop=True)
df_unlabeled['text_slang_removal'] = df_unlabeled['text_slang_removal'].astype(str)

print(f"Dataset Overview:")
print(f"  Total data asli: {len(df_asli)}")
print(f"  Data sudah berlabel: {len(df_labeled)}")
print(f"  Data unlabeled: {len(df_unlabeled)}")
print(f"\nTarget: Labeling {len(df_unlabeled)} data otomatis...\n")

def tokenize_function(examples):
    return tokenizer(examples["text_slang_removal"], padding="max_length",
                     truncation=True, max_length=128)

unlabeled_ds = Dataset.from_pandas(df_unlabeled[['text_slang_removal']])
unlabeled_tokenized = unlabeled_ds.map(tokenize_function, batched=True)

print("IndoBERT sedang bekerja melabeli data...")
print("Mohon tunggu sekitar 1-2 menit...\n")

training_args = TrainingArguments(
    output_dir="./temp_prediction",
    per_device_eval_batch_size=16,
)

trainer = Trainer(
    model=model,
    args=training_args,
)

raw_preds = trainer.predict(unlabeled_tokenized)
predicted_labels = np.argmax(raw_preds.predictions, axis=1)

df_unlabeled['label'] = predicted_labels

print("Prediksi selesai!\n")

df_labeled['label'] = df_labeled['label'].astype(int)
df_unlabeled['label'] = df_unlabeled['label'].astype(int)

df_total_final = pd.concat([df_labeled, df_unlabeled], ignore_index=True)

print(f"Total data setelah labeling: {len(df_total_final)}")

label_map = {0: 'Negatif', 1: 'Netral', 2: 'Positif'}

unlabeled_counts = df_unlabeled['label'].map(label_map).value_counts()
print(f"\n🔍 Distribusi data yang baru dilabeli ({len(df_unlabeled)} data):")
for sentiment, count in unlabeled_counts.items():
    pct = (count / len(df_unlabeled)) * 100
    print(f"  {sentiment:8} : {count:4} ({pct:.1f}%)")

final_counts = df_total_final['label'].map(label_map).value_counts()
print(f"\nDistribusi sentimen TOTAL ({len(df_total_final)} data):")
for sentiment, count in final_counts.items():
    pct = (count / len(df_total_final)) * 100
    print(f"  {sentiment:8} : {count:4} ({pct:.1f}%)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors = ['#ff9999', '#66b3ff', '#99ff99']
ax1.pie(unlabeled_counts, labels=unlabeled_counts.index, autopct='%1.1f%%',
        startangle=140, colors=colors, explode=(0.05, 0.05, 0.05), shadow=True)
ax1.set_title(f'Distribusi Sentimen\nData Baru Dilabeli ({len(df_unlabeled)} data)',
              fontsize=13, fontweight='bold')

ax2.pie(final_counts, labels=final_counts.index, autopct='%1.1f%%',
        startangle=140, colors=colors, explode=(0.05, 0.05, 0.05), shadow=True)
ax2.set_title(f'Distribusi Sentimen Total\nPublik Terhadap Polri ({len(df_total_final)} data)',
              fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('distribusi_sentimen_final.png', dpi=300, bbox_inches='tight')
plt.show()

output_file = 'HASIL_KLASIFIKASI_TOTAL_INDOBERT_FINISH.csv'
df_total_final.to_csv(output_file, index=False)

print("\n" + "="*60)
print("AUTO-LABELING SELESAI!")
print("="*60)
print(f"\nFile yang dihasilkan:")
print(f"  1. {output_file}")
print(f"  2. distribusi_sentimen_final.png")
print(f"\n{len(df_unlabeled)} data berhasil dilabeli otomatis!")
print(f"Total dataset final: {len(df_total_final)} data")
print("\nGunakan untuk analisis sentimen di Bab IV!")
print("="*60)

Memuat 'Senjata' (Model 85% Akurasi)...


HFValidationError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: './model_sentimen_polri'.